# Diabetic Retinopathy Classification - Step by Step Training

This notebook allows you to train and evaluate your CNN models step by step.

In [ ]:
import os
import sys
import torch
import torch.nn as nn
import torch.optim as optim

# Add parent directory to path to import local modules
sys.path.append('..')
from data.dataset import prepare_data
from models.cnn_models import get_model
from training.trainer import train_model
from evaluation.evaluator import evaluate_model, plot_confusion_matrix, plot_training_history

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

## 1. Prepare Data
Load the images, apply splits, and prepare DataLoaders.

In [ ]:
DATASET_DIR = "../dataset"
BATCH_SIZE = 32

print("Preparing data...")
train_loader, val_loader, test_loader, num_classes = prepare_data(DATASET_DIR, batch_size=BATCH_SIZE)

## 2. Initialize Model
Choose a model to train: `resnet18`, `resnet50`, `efficientnet_b0`, `densenet121`, or `mobilenet_v3_large`.

In [ ]:
model_name = 'resnet18' # <--- Change this to train a different model

model = get_model(model_name, num_classes)
model = model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.0001)

## 3. Train Model

In [ ]:
EPOCHS = 25
PATIENCE = 5

# Note: logs and checkpoints will be saved relative to where the notebook is running
# So we point them to the parent directory
model, history = train_model(
    model=model,
    model_name=model_name,
    train_loader=train_loader,
    val_loader=val_loader,
    criterion=criterion,
    optimizer=optimizer,
    device=device,
    epochs=EPOCHS,
    patience=PATIENCE,
    save_dir='../checkpoints',
    logs_dir='../logs'
)

## 4. Evaluate Model

In [ ]:
acc, precision, recall, f1, y_true, y_pred = evaluate_model(model, test_loader, device)
print(f"Test Acc: {acc:.4f}, Precision: {precision:.4f}, Recall: {recall:.4f}, F1: {f1:.4f}")

classes = [str(i) for i in range(num_classes)]
plot_confusion_matrix(y_true, y_pred, classes, model_name, save_dir='../outputs')
plot_training_history(f'../logs/{model_name}_history.json', model_name, save_dir='../outputs')